# 14_Career_Advisor

RAG-powered career advisor that answers natural-language queries about skills.

### Capabilities
- "Should I learn Tableau?"
- "What skills should a Data Analyst learn?"
- "What skills are becoming obsolete?"
- "Compare Python vs R for a data science career"
- "What are the safest skills to learn in Cloud?"

### Architecture
```
User Query
    ↓
FAISS Semantic Retrieval  (top-k skill documents)
    ↓
Context Assembly
    ↓
LLM Generation  (Claude / local model)
    ↓
Structured Answer
```

### Outputs
- `career_advisor_demo.json` — sample Q&A pairs
- `career_advisor.py`        — importable advisor class


In [9]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ── load knowledge base ───────────────────────────────────────────────────────
KB_PATH   = '../Generated Datasets/skill_knowledge_base.json'
FAISS_PATH = '../Generated Datasets/faiss_index.bin'
META_PATH  = '../Generated Datasets/faiss_metadata.json'

with open(KB_PATH) as f:
    knowledge_base = json.load(f)

# index by skill name for O(1) lookup
KB_INDEX = {doc['skill']: doc for doc in knowledge_base}

print(f"Knowledge base loaded: {len(knowledge_base)} skills")
print(f"Sample keys: {list(KB_INDEX.keys())[:8]}")


Knowledge base loaded: 114 skills
Sample keys: ['javascript', 'python', 'sql', 'postgresql', 'docker', 'typescript', 'data analysis', 'aws']


In [10]:
# ── FAISS retriever ───────────────────────────────────────────────────────────
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("faiss-cpu not installed — keyword fallback retriever will be used.")

if FAISS_AVAILABLE:
    faiss_index = faiss.read_index(FAISS_PATH)
    with open(META_PATH) as f:
        faiss_meta = json.load(f)

    # Align knowledge-base order with FAISS metadata
    VECTOR_COLS = [
        'linkedin_demand', 'salary_premium', 'current_usage',
        'future_interest', 'growth_rate', 'global_adoption', 'ses_score',
    ]

    def _get_query_vec(skill_name=None, feature_dict=None):
        if skill_name and skill_name in KB_INDEX:
            doc = KB_INDEX[skill_name]
        elif feature_dict:
            doc = feature_dict
        else:
            return None
        vec = np.array([[doc.get(c, 0.0) or 0.0 for c in VECTOR_COLS]], dtype=np.float32)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def faiss_retrieve(query_skill=None, feature_dict=None, k=6):
        vec = _get_query_vec(query_skill, feature_dict)
        if vec is None:
            return []
        D, I = faiss_index.search(vec, k)
        results = []
        for dist, idx in zip(D[0], I[0]):
            meta  = faiss_meta[idx]
            skill = meta['skill']
            if skill in KB_INDEX:
                results.append({'score': float(dist), 'doc': KB_INDEX[skill]})
        return results


In [11]:
# ── keyword fallback retriever ────────────────────────────────────────────────
def keyword_retrieve(query: str, k: int = 6):
    """Simple BM25-lite: match query tokens against skill names + narratives."""
    tokens = set(query.lower().split())
    scored = []
    for doc in knowledge_base:
        text  = f"{doc['skill']} {doc.get('sub_category','')} {doc.get('narrative','')}".lower()
        score = sum(1 for t in tokens if t in text)
        # boost exact skill name match
        if doc['skill'] in query.lower():
            score += 10
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [{'score': s, 'doc': d} for s, d in scored[:k]]

def retrieve(query: str, anchor_skill: str = None, k: int = 6):
    """Unified retrieval: FAISS if available, else keyword."""
    if FAISS_AVAILABLE and anchor_skill and anchor_skill in KB_INDEX:
        return faiss_retrieve(query_skill=anchor_skill, k=k)
    return keyword_retrieve(query, k=k)


In [12]:
# ── context builder ───────────────────────────────────────────────────────────
def build_context(retrieved_docs: list) -> str:
    """Format retrieved skill documents into a compact LLM context string."""
    lines = []
    for item in retrieved_docs:
        doc = item['doc']
        lines.append(
            f"SKILL: {doc['skill'].title()}\n"
            f"  Category   : {doc.get('sub_category','?')}\n"
            f"  SES Score  : {doc.get('ses_score','?')} ({doc.get('ses_tier','?')})\n"
            f"  Archetype  : {doc.get('archetype','?')}\n"
            f"  Industries : {', '.join(doc.get('industries', [])) or 'General'}\n"
            f"  Forecast   : 1yr={doc.get('forecast_1y','?')}  "
            f"3yr={doc.get('forecast_3y','?')}  "
            f"Trend={doc.get('forecast_trend','?')}\n"
            f"  Summary    : {doc.get('narrative','')[:220]}\n"
        )
    return "\n".join(lines)


In [13]:
# ── system prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are HSEP Career Advisor — a precise, data-driven career intelligence assistant.

You have access to a skill intelligence database covering SES scores, demand forecasts,
industry relevance, and archetype classifications for hundreds of technical skills.

Rules:
- Ground every recommendation in the provided skill context.
- Cite SES scores, tiers, and forecast trends when relevant.
- Be concise but specific. Avoid generic career advice.
- If the user asks to compare skills, structure the answer as a comparison table.
- Always end with a one-line actionable recommendation.
"""


In [14]:
# ── LLM answer generation ─────────────────────────────────────────────────────
import urllib.request

OPENROUTER_API_KEY = "API KEY"
OPENROUTER_MODEL   = "openai/gpt-oss-120b:free"
OPENROUTER_URL     = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(messages: list, system: str = SYSTEM_PROMPT) -> str:
    """
    Call OpenRouter (openai/gpt-oss-120b:free) via REST.
    Falls back to a rule-based message if the API is unavailable.
    """
    # Convert Anthropic-style messages to OpenAI-style (system injected as first message)
    oai_messages = [{"role": "system", "content": system}] + [
        {"role": m["role"], "content": m["content"]} for m in messages
    ]

    payload = json.dumps({
        "model"      : OPENROUTER_MODEL,
        "max_tokens" : 1000,
        "messages"   : oai_messages,
    }).encode()

    req = urllib.request.Request(
        OPENROUTER_URL,
        data    = payload,
        method  = "POST",
        headers = {
            "Content-Type"  : "application/json",
            "Authorization" : f"Bearer {OPENROUTER_API_KEY}",
            "HTTP-Referer"  : "https://hsep.app",
            "X-Title"       : "HSEP Career Advisor",
        },
    )

    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            body = json.loads(resp.read())
            return body["choices"][0]["message"]["content"]
    except Exception as e:
        return f"[LLM unavailable: {e}]\nCheck your OPENROUTER_API_KEY or model name."


In [15]:
# ── main advisor function ─────────────────────────────────────────────────────
def career_advisor(query: str, verbose: bool = True) -> dict:
    """
    Full RAG pipeline: retrieve → context → generate → return.

    Parameters
    ----------
    query   : natural-language career question
    verbose : print intermediate steps

    Returns
    -------
    dict with keys: query, retrieved_skills, context, answer
    """
    if verbose:
        print(f"Query: {query}\n{'─'*60}")

    # 1. Extract anchor skill from query (simple heuristic)
    query_lower = query.lower()
    anchor = next(
        (doc['skill'] for doc in knowledge_base if doc['skill'] in query_lower),
        None
    )

    # 2. Retrieve
    retrieved = retrieve(query, anchor_skill=anchor, k=6)
    retrieved_skills = [r['doc']['skill'] for r in retrieved]

    if verbose:
        print(f"Retrieved skills: {retrieved_skills}")

    # 3. Build context
    context = build_context(retrieved)

    # 4. Build messages
    messages = [{
        "role"   : "user",
        "content": (
            f"Skill Intelligence Context:\n{context}\n\n"
            f"Career Question: {query}"
        )
    }]

    # 5. Generate answer
    answer = call_llm(messages)

    if verbose:
        print(f"\nAnswer:\n{answer}")

    return {
        "query"            : query,
        "anchor_skill"     : anchor,
        "retrieved_skills" : retrieved_skills,
        "context_preview"  : context[:400] + "...",
        "answer"           : answer,
    }


In [16]:
# ── demo queries ─────────────────────────────────────────────────────────────
DEMO_QUERIES = [
    "Should I learn Tableau or Power BI for a data analytics career?",
    "What skills are at risk of becoming obsolete in the next 3 years?",
    "What are the top skills I should learn for a career in AI/ML?",
    "Is Kubernetes still worth learning in 2025?",
    "What skills should a junior Data Engineer focus on?",
    "Compare Python vs R for a data science career",
    "What are the safest cloud skills to invest in right now?",
]

demo_results = []
for q in DEMO_QUERIES:
    print("\n" + "="*70)
    result = career_advisor(q, verbose=True)
    demo_results.append(result)
    print()



Query: Should I learn Tableau or Power BI for a data analytics career?
────────────────────────────────────────────────────────────
Retrieved skills: ['data analytics', 'tableau', 'agile development', 'agile methodologies', 'agile methodology', 'automation']

Answer:
**Comparison (based on available data)**  

| Skill | Category | SES Score | Archetype | Primary Industries | Forecast Trend |
|------|----------|-----------|-----------|--------------------|----------------|
| **Tableau** | Data Analytics | **0.3253** (Safe) | Growing | Data Science, Data Analytics | Unknown |
| **Power BI** | – (no record in the current database) | – | – | – | – |

**Interpretation**

* Tableau is a **safe** skill with a solid SES score (0.3253) and belongs to the **Growing** archetype, indicating stable demand across data‑science and analytics roles.  
* Power BI lacks representation in the current skill‑intelligence set, so we cannot provide an SES score or trend. While Power BI is widely used in the 

In [17]:
# ── save demo Q&A ─────────────────────────────────────────────────────────────
demo_path = '../Generated Datasets/career_advisor_demo.json'
with open(demo_path, 'w') as f:
    json.dump(demo_results, f, indent=2)

print(f"Demo Q&A saved → {demo_path}")
print(f"Total queries answered: {len(demo_results)}")


Demo Q&A saved → ../Generated Datasets/career_advisor_demo.json
Total queries answered: 7


In [18]:
ADVISOR_MODULE = '''
"""
career_advisor.py — HSEP Career Advisor (importable module)
Usage:
    from career_advisor import career_advisor
    result = career_advisor("Should I learn Kubernetes?")
    print(result["answer"])
"""
import json, numpy as np, urllib.request

KB_PATH   = "skill_knowledge_base.json"
META_PATH = "faiss_metadata.json"

with open(KB_PATH) as f:
    knowledge_base = json.load(f)
KB_INDEX = {d["skill"]: d for d in knowledge_base}

try:
    import faiss
    FAISS_INDEX = faiss.read_index("faiss_index.bin")
    with open(META_PATH) as f:
        FAISS_META = json.load(f)
    FAISS_AVAILABLE = True
except Exception:
    FAISS_AVAILABLE = False

VECTOR_COLS = [
    "linkedin_demand","salary_premium","current_usage",
    "future_interest","growth_rate","global_adoption","ses_score",
]

SYSTEM_PROMPT = (
    "You are HSEP Career Advisor — a precise, data-driven career "
    "intelligence assistant. Ground every recommendation in the provided "
    "skill context. Cite SES scores, tiers, and forecast trends when relevant."
)

OPENROUTER_API_KEY = "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"
OPENROUTER_MODEL   = "openai/gpt-oss-120b:free"
OPENROUTER_URL     = "https://openrouter.ai/api/v1/chat/completions"

def _keyword_retrieve(query, k=6):
    tokens = set(query.lower().split())
    scored = []
    for doc in knowledge_base:
        text  = f"{doc['skill']} {doc.get('sub_category','')} {doc.get('narrative','')}".lower()
        score = sum(1 for t in tokens if t in text) + (10 if doc["skill"] in query.lower() else 0)
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [{"score": s, "doc": d} for s, d in scored[:k]]

def _faiss_retrieve(anchor, k=6):
    if anchor not in KB_INDEX:
        return []
    doc = KB_INDEX[anchor]
    vec = np.array([[doc.get(c, 0.0) or 0.0 for c in VECTOR_COLS]], dtype=np.float32)
    norm = np.linalg.norm(vec)
    if norm > 0: vec /= norm
    D, I = FAISS_INDEX.search(vec, k)
    return [{"score": float(d), "doc": KB_INDEX[FAISS_META[i]["skill"]]}
            for d, i in zip(D[0], I[0]) if FAISS_META[i]["skill"] in KB_INDEX]

def _build_context(retrieved):
    lines = []
    for item in retrieved:
        d = item["doc"]
        lines.append(
            f"SKILL: {d['skill'].title()}\n"
            f"  SES: {d.get('ses_score','?')} ({d.get('ses_tier','?')}) | "
            f"Archetype: {d.get('archetype','?')} | "
            f"Trend: {d.get('forecast_trend','?')}\n"
            f"  {d.get('narrative','')[:200]}\n"
        )
    return "\n".join(lines)

def career_advisor(query: str) -> dict:
    anchor    = next((d["skill"] for d in knowledge_base if d["skill"] in query.lower()), None)
    retrieved = (_faiss_retrieve(anchor) if FAISS_AVAILABLE and anchor
                 else _keyword_retrieve(query))
    context   = _build_context(retrieved)

    oai_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    payload = json.dumps({
        "model"     : OPENROUTER_MODEL,
        "max_tokens": 1000,
        "messages"  : oai_messages,
    }).encode()
    req = urllib.request.Request(
        OPENROUTER_URL, data=payload, method="POST",
        headers={
            "Content-Type" : "application/json",
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "HTTP-Referer" : "https://hsep.app",
            "X-Title"      : "HSEP Career Advisor",
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            body   = json.loads(resp.read())
            answer = body["choices"][0]["message"]["content"]
    except Exception as e:
        answer = f"[API error: {e}]"
    return {"query": query,
            "retrieved_skills": [r["doc"]["skill"] for r in retrieved],
            "answer": answer}
'''

module_path = '../Generated Datasets/career_advisor.py'
with open(module_path, 'w') as f:
    f.write(ADVISOR_MODULE)

print(f"Module saved → {module_path}")


Module saved → ../Generated Datasets/career_advisor.py


In [19]:
print("=" * 60)
print("Career Advisor — Build Summary")
print("=" * 60)
print(f"  Knowledge base : {len(knowledge_base)} skills")
print(f"  FAISS available: {FAISS_AVAILABLE}")
print(f"  Demo Q&A       : career_advisor_demo.json")
print(f"  Module         : career_advisor.py")
print()
print("Quick-start:")
print('  from career_advisor import career_advisor')
print('  print(career_advisor("Should I learn Kubernetes?")["answer"])')


Career Advisor — Build Summary
  Knowledge base : 114 skills
  FAISS available: True
  Demo Q&A       : career_advisor_demo.json
  Module         : career_advisor.py

Quick-start:
  from career_advisor import career_advisor
  print(career_advisor("Should I learn Kubernetes?")["answer"])
